# Como o Senado vota? Análise das votações nominais de 2025

Este notebook extrai, trata e analisa dados da
[API de Dados Abertos do Senado Federal](https://legis.senado.leg.br/dadosabertos/api-docs/swagger-ui/index.html),
com foco nas **votações nominais do Plenário em 2025**.

## 1. Configuração

Bibliotecas e parâmetros usados em todo o notebook. `PAUSA` espaça as requisições para não
sobrecarregar a API pública.

In [1]:
import logging
import time
from typing import Any

import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

ANO: int = 2025
BASE_URL: str = "https://legis.senado.leg.br/dadosabertos"
TIMEOUT: int = 30   # segundos por requisição
PAUSA: float = 0.2  # segundos entre requisições

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s", force=True)
log = logging.getLogger("senado")

## 2. Cliente da API

Ao explorar a API, encontramos quatro comportamentos que o cliente precisa tratar:

| Comportamento observado | Tratamento |
|---|---|
| Sem o cabeçalho `Accept: application/json`, a API responde em **XML** | A sessão envia o cabeçalho em toda requisição |
| Um código inexistente (ex.: `/senador/99999999`) retorna **HTTP 200** só com metadados, sem erro | `extrair()` devolve lista vazia quando o caminho não existe |
| Cada endpoint aninha os dados de um jeito (`ListaParlamentarEmExercicio → Parlamentares → Parlamentar`, lista simples em `/votacao`...) e, por herança do XML, **um item único pode vir como objeto em vez de lista** | `extrair()` percorre o caminho informado e sempre devolve `list[dict]` |
| Endpoints documentados que respondem **404** (ex.: `/plenario/votacao/orientacaoBancada/{inicio}/{fim}`) | Erros HTTP viram `ErroAPI`, com mensagem clara |

Além disso, a sessão faz até 5 novas tentativas com *backoff* exponencial em erros transitórios
(`429` e `5xx`).

In [2]:
class ErroAPI(Exception):
    """Falha ao consultar a API do Senado (rede, HTTP ou resposta inválida)."""


def criar_sessao() -> requests.Session:
    sessao = requests.Session()
    retry = Retry(
        total=5,
        backoff_factor=1,  
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"],
        respect_retry_after_header=True,
    )
    sessao.mount("https://", HTTPAdapter(max_retries=retry))
    sessao.headers.update({"Accept": "application/json", "User-Agent": "desafio-aleon/1.0"})
    return sessao


def get_json(sessao: requests.Session, endpoint: str, params: dict[str, Any] | None = None) -> Any:
    url = f"{BASE_URL}{endpoint}"
    try:
        resposta = sessao.get(url, params=params, timeout=TIMEOUT)
        resposta.raise_for_status()
        return resposta.json()
    except requests.exceptions.Timeout as e:
        raise ErroAPI(f"Timeout após {TIMEOUT}s: {url}") from e
    except requests.exceptions.RetryError as e:
        raise ErroAPI(f"Tentativas esgotadas: {url}") from e
    except requests.exceptions.ConnectionError as e:
        raise ErroAPI(f"Falha de conexão: {url}") from e
    except requests.exceptions.HTTPError as e:
        raise ErroAPI(f"HTTP {e.response.status_code}: {url}") from e
    except ValueError as e:  
        raise ErroAPI(f"Resposta não é JSON válido: {url}") from e
    finally:
        time.sleep(PAUSA)


def extrair(payload: Any, *caminho: str) -> list[dict]:
    
    no = payload
    for chave in caminho:
        if not isinstance(no, dict) or chave not in no:
            return []
        no = no[chave]
    if no is None:
        return []
    return no if isinstance(no, list) else [no]

### 2.1 Validação do cliente

Dois testes dos casos de borda descritos acima. Os casos normais são exercitados pela própria
extração, na seção 3.

In [3]:
sessao = criar_sessao()

# Código inexistente: HTTP 200 sem dados -> lista vazia, sem quebrar
inexistente = extrair(get_json(sessao, "/senador/99999999"), "DetalheParlamentar", "Parlamentar")
print(f"Senador inexistente: {inexistente}")

# Endpoint documentado que responde 404 -> ErroAPI tratado
try:
    get_json(sessao, f"/plenario/votacao/orientacaoBancada/{ANO}-03-01/{ANO}-03-31")
except ErroAPI as e:
    print(f"Erro tratado: {e}")

Senador inexistente: []


Erro tratado: HTTP 404: https://legis.senado.leg.br/dadosabertos/plenario/votacao/orientacaoBancada/2025-03-01/2025-03-31


## 3. Extração

### 3.1 Seleção dos dados

A pergunta central é **como o Senado vota**: quem vota com quem e como cada grupo reage às
propostas de outros grupos. Para isso, extraímos três conjuntos:

| Conjunto | Endpoint | Por que |
|---|---|---|
| Votações nominais de 2025, com o voto de cada senador | `/votacao` | Base de toda a análise; **uma única chamada** traz as 128 votações e os votos |
| Senadores em exercício | `/senador/lista/atual` | Perfil (sexo, UF) e **bloco parlamentar** de cada partido |
| Autoria das propostas votadas | `/processo/{id}` | Quem propôs cada matéria (Governo, Câmara, Senado) e de qual partido |

**Dados avaliados e descartados:**

- **Licenças** (`/senador/{codigo}/licencas`): seriam 81 chamadas, mas são redundantes. O próprio
  registro de cada voto já informa o motivo da ausência (licença saúde, missão oficial, não compareceu).
- **Filiações partidárias**: o voto já traz o partido do senador **na data da votação**, o que basta
  para identificar trocas de partido ao longo do ano.
- **Projetos de lei e discursos**: fogem do foco nas votações.

### 3.2 Funções de extração

Falhas pontuais (a autoria de uma proposta) são registradas em `falhas` e **não interrompem** a
extração. Já as votações e os senadores são essenciais: se falharem, a extração é abortada.

In [4]:
falhas: list[dict] = []  # reiniciada a cada execução desta célula


def registrar_falha(etapa: str, chave: Any, erro: Exception) -> None:
    """Guarda uma falha não fatal para o relatório ao fim da extração."""
    log.warning("Falha em %s (%s): %s", etapa, chave, erro)
    falhas.append({"etapa": etapa, "chave": chave, "erro": str(erro)})


def extrair_votacoes(sessao: requests.Session, ano: int) -> list[dict]:
    """Votações nominais do Plenário no ano, cada uma com a lista de votos dos senadores."""
    votacoes = get_json(sessao, "/votacao", {"dataInicio": f"{ano}-01-01", "dataFim": f"{ano}-12-31"})
    if not isinstance(votacoes, list):
        raise ErroAPI(f"Formato inesperado em /votacao: {type(votacoes).__name__}")
    return votacoes


def extrair_senadores(sessao: requests.Session) -> list[dict]:
    """Senadores em exercício, com partido, UF e bloco parlamentar."""
    payload = get_json(sessao, "/senador/lista/atual")
    return extrair(payload, "ListaParlamentarEmExercicio", "Parlamentares", "Parlamentar")


def extrair_autorias(sessao: requests.Session, ids_processo: list[int]) -> dict[int, list[dict]]:
    """Autores (de iniciativa) de cada processo legislativo, indexados pelo id do processo."""
    autorias: dict[int, list[dict]] = {}
    for i, id_processo in enumerate(ids_processo, 1):
        try:
            autorias[id_processo] = get_json(sessao, f"/processo/{id_processo}").get("autoriaIniciativa") or []
        except (ErroAPI, AttributeError) as e:
            registrar_falha("autoria", id_processo, e)
        if i % 25 == 0:
            log.info("Autorias: %d/%d", i, len(ids_processo))
    return autorias

### 3.3 Execução

A extração completa leva cerca de 1 minuto (uma chamada por proposta votada).

In [5]:
try:
    votacoes_raw = extrair_votacoes(sessao, ANO)
    senadores_raw = extrair_senadores(sessao)
except ErroAPI as e:
    log.error("Falha em dado essencial, abortando: %s", e)
    raise

ids_processo = sorted({v["idProcesso"] for v in votacoes_raw if v.get("idProcesso")})
autorias_raw = extrair_autorias(sessao, ids_processo)

print(f"Votações: {len(votacoes_raw)} | Senadores em exercício: {len(senadores_raw)} | "
      f"Autorias: {len(autorias_raw)}/{len(ids_processo)} processos | Falhas: {len(falhas)}")

2026-11-07 17:16:32,782 [INFO] Autorias: 25/111


2026-11-07 17:16:38,868 [INFO] Autorias: 50/111


2026-11-07 17:16:44,959 [INFO] Autorias: 75/111


2026-11-07 17:16:51,033 [INFO] Autorias: 100/111


Votações: 128 | Senadores em exercício: 81 | Autorias: 111/111 processos | Falhas: 0


## 4. Tratamento

A API devolve JSON aninhado, com números e booleanos em texto (`"S"`/`"N"`, `"Sim"`/`"Não"`). O
tratamento tem duas camadas:

1. **Pipeline genérico de limpeza** (`limpar`): funciona em qualquer `DataFrame` e cuida do que não
   depende do assunto: nomes de colunas, seleção, remoção do irrelevante, conversão de tipos e duplicatas.
2. **Regras do domínio**: o que só faz sentido para o Senado, como agrupar os códigos de voto, atribuir
   blocos parlamentares e classificar a autoria das propostas.

### 4.1 Pipeline genérico de limpeza

| Etapa | O que faz |
|---|---|
| 1. Nomes de colunas | `IdentificacaoParlamentar.CodigoParlamentar` → `identificacao_parlamentar_codigo_parlamentar` |
| 2. Remoção do irrelevante | Colunas fora da seleção, indicadas (aceita padrões como `informe_legislativo_*`), aninhadas, quase vazias ou constantes; o motivo vai para o relatório |
| 3. Textos | Remove espaços extras e troca marcadores de vazio (`""`, `"-"`, `"null"`...) por nulo |
| 4. Tipos | Detecta booleano (`S`/`N`, `Sim`/`Não`), número (inclusive `1.234,56`), data e inteiros com nulo |
| 5. Duplicatas | Pela linha inteira ou por uma chave |
| 6. Nulos | Preenchimento opcional, por coluna |
| 7. Relatório | Lista de tudo o que foi alterado |

**Cuidado com regras genéricas:** o marcador de vazio `"na"` coincide com o código de voto `NA`
(*dispositivo não citado*). Sem proteção, 15 votos virariam nulos. Por isso `manter_texto` impede que
colunas de código sejam alteradas.

In [6]:
import re
import unicodedata
import warnings
from fnmatch import fnmatch

import pandas as pd

VALORES_VAZIOS: set[str] = {"", "-", "--", "n/a", "na", "nan", "null", "none", "nd", "s/n", "?"}
VERDADEIRO: set[str] = {"sim", "s", "true", "t", "yes", "y", "1"}
FALSO: set[str] = {"nao", "não", "n", "false", "f", "no", "0"}


def padronizar_nome(nome: str) -> str:
    """'IdentificacaoParlamentar.CodigoParlamentar' -> 'identificacao_parlamentar_codigo_parlamentar'."""
    nome = re.sub(r"(?<=[a-z0-9])(?=[A-Z])", "_", str(nome))  # camelCase -> snake_case
    nome = re.sub(r"(?<=[A-Z])(?=[A-Z][a-z])", "_", nome)      # siglaUFParlamentar -> sigla_UF_Parlamentar
    nome = unicodedata.normalize("NFKD", nome).encode("ascii", "ignore").decode()  # remove acentos
    return re.sub(r"[^0-9a-zA-Z]+", "_", nome).strip("_").lower() or "coluna"


def _casa(coluna: str, padroes: list[str]) -> bool:
    """True se a coluna casa com algum padrão glob (ex.: 'informe_legislativo_*')."""
    return any(fnmatch(coluna, p) for p in padroes)


def remover_irrelevantes(df: pd.DataFrame, selecionar: list[str], remover: list[str],
                         max_nulos: float, log: list[str]) -> pd.DataFrame:
    """Remove colunas fora da seleção, indicadas, aninhadas, quase vazias ou constantes."""
    motivos: dict[str, str] = {}
    for col in df.columns:
        if selecionar and not _casa(col, selecionar):
            motivos[col] = "fora da seleção"
        elif _casa(col, remover):
            motivos[col] = "indicada"
        elif df[col].map(lambda v: isinstance(v, (list, dict))).any():
            motivos[col] = "estrutura aninhada"  # listas/objetos que o json_normalize não achatou
        elif df[col].isna().mean() > max_nulos:
            motivos[col] = f"{df[col].isna().mean():.0%} nulos"
        elif df[col].nunique(dropna=True) <= 1:
            motivos[col] = f"valor constante ({df[col].dropna().iloc[0] if df[col].notna().any() else 'vazio'})"

    fora = [c for c, m in motivos.items() if m == "fora da seleção"]
    if fora:
        log.append(f"removidas {len(fora)} colunas fora da seleção")
    log += [f"removida '{c}': {m}" for c, m in motivos.items() if m != "fora da seleção"]
    return df.drop(columns=list(motivos)).dropna(how="all")


def normalizar_textos(df: pd.DataFrame, manter_texto: list[str]) -> pd.DataFrame:
    """Remove espaços extras e transforma marcadores de 'vazio' em NA.

    Colunas em `manter_texto` não têm marcadores trocados: no Senado, o voto 'NA' é um código
    válido ('dispositivo não citado') e viraria nulo.
    """
    for col in df.select_dtypes(include=["object", "string"]).columns:
        s = df[col].astype("string").str.strip().str.replace(r"\s+", " ", regex=True)
        df[col] = s if col in manter_texto else s.mask(s.str.lower().isin(VALORES_VAZIOS))
    return df


def _parece_identificador(s: pd.Series) -> bool:
    """Códigos com zero à esquerda (CNPJ, CEP, CPF) não devem virar número."""
    amostra = s.dropna().astype(str).head(500)
    return bool(len(amostra)) and amostra.str.match(r"^0\d+").mean() > 0.05


def _para_numero(s: pd.Series) -> pd.Series:
    """Aceita '1.234,56', '1234.56', 'R$ 10' e '15%'."""
    txt = s.astype("string").str.replace(r"[R$%\s]", "", regex=True)
    br = txt.str.contains(r",\d{1,2}$", na=False)  # formato brasileiro
    txt = txt.where(~br, txt.str.replace(".", "", regex=False).str.replace(",", ".", regex=False))
    return pd.to_numeric(txt, errors="coerce")


def converter_tipos(df: pd.DataFrame, manter_texto: list[str], limiar: float,
                    log: list[str]) -> pd.DataFrame:
    """Converte cada coluna para o tipo aceito por >= `limiar` dos valores não nulos."""
    # decimais que só têm inteiros (ex.: contagens com nulo, que o pandas lê como float)
    for col in df.select_dtypes(include="float").columns:
        if (df[col].dropna() % 1 == 0).all():
            df[col] = df[col].astype("Int64")
            log.append(f"'{col}' -> inteiro (com nulos)")

    for col in df.select_dtypes(include=["object", "string"]).columns:
        s = df[col]
        validos = s.notna().sum()
        if col in manter_texto or validos == 0 or _parece_identificador(s):
            continue

        if s.dropna().str.lower().isin(VERDADEIRO | FALSO).mean() >= limiar:
            df[col] = s.str.lower().map(lambda v: True if v in VERDADEIRO
                                        else False if v in FALSO else pd.NA).astype("boolean")
            log.append(f"'{col}' -> booleano")
            continue

        num = _para_numero(s)
        if num.notna().sum() / validos >= limiar:
            inteiro = (num.dropna() % 1 == 0).all()
            df[col] = num.astype("Int64") if inteiro else num
            log.append(f"'{col}' -> {'inteiro' if inteiro else 'decimal'}")
            continue

        with warnings.catch_warnings():  # o pandas avisa quando infere o formato elemento a elemento
            warnings.simplefilter("ignore", UserWarning)
            iso = s.str.match(r"^\d{4}-").fillna(False).any()
            datas = pd.to_datetime(s, errors="coerce", dayfirst=not iso)
        if datas.notna().sum() / validos >= limiar:
            df[col] = datas
            log.append(f"'{col}' -> data")
    return df


def limpar(df: pd.DataFrame, *, colunas: dict[str, str] | None = None, remover: list[str] | None = None,
           manter_texto: list[str] | None = None,
           chave_duplicata: list[str] | None = None, preencher: dict[str, Any] | None = None,
           max_nulos: float = 0.95, limiar_tipo: float = 0.9) -> tuple[pd.DataFrame, list[str]]:
    """Limpa um DataFrame qualquer e devolve (df_limpo, relatorio).

    Os parâmetros usam os nomes de coluna já padronizados (snake_case).

    colunas          {nome_padronizado: nome_final}: mantém só essas colunas e as renomeia no fim
                     (padrão: todas, com o nome padronizado)
    remover          padrões glob das colunas a descartar (ex.: 'informe_legislativo*')
    manter_texto     colunas que nunca são convertidas nem têm vazios trocados (códigos, IDs)
    chave_duplicata  colunas que definem duplicata (padrão: linha inteira)
    preencher        {coluna: valor} para preencher nulos
    max_nulos        remove colunas com proporção de nulos acima disso
    limiar_tipo      fração mínima de valores válidos para aceitar uma conversão
    """
    log: list[str] = [f"entrada: {df.shape[0]} linhas x {df.shape[1]} colunas"]
    df = df.copy()
    df.columns = [padronizar_nome(c) for c in df.columns]
    manter_texto = manter_texto or []

    df = remover_irrelevantes(df, list(colunas or {}), remover or [], max_nulos, log)
    df = normalizar_textos(df, manter_texto)
    df = converter_tipos(df, manter_texto, limiar_tipo, log)

    antes = len(df)
    df = df.drop_duplicates(subset=chave_duplicata).reset_index(drop=True)
    if antes - len(df):
        log.append(f"removidas {antes - len(df)} linhas duplicadas")
    if preencher:
        df = df.fillna(preencher)
    if colunas:
        df = df.rename(columns=colunas)

    log.append(f"saída: {df.shape[0]} linhas x {df.shape[1]} colunas")
    return df, log


def categorizar(df: pd.DataFrame, max_categorias: int = 50) -> pd.DataFrame:
    """Texto com poucos valores distintos vira 'category' (menos memória, ordenação explícita)."""
    for col in df.select_dtypes(include=["object", "string"]).columns:
        if df[col].nunique() <= max_categorias:
            df[col] = df[col].astype("category")
    return df


def resumo(df: pd.DataFrame) -> pd.DataFrame:
    """Diagnóstico por coluna: tipo, % de nulos, nº de valores distintos e um exemplo."""
    return pd.DataFrame({
        "tipo": df.dtypes.astype(str),
        "nulos_%": (df.isna().mean() * 100).round(1),
        "distintos": df.nunique(),
        "exemplo": df.apply(lambda c: c.dropna().iloc[0] if c.notna().any() else None),
    })

In [7]:
def mostrar(relatorio: list[str]) -> None:
    """Imprime o relatório de limpeza."""
    print("\n".join(f"  · {linha}" for linha in relatorio), end="\n\n")

### 4.2 Senadores e blocos parlamentares

Da listagem de senadores usamos só **código, nome, partido e bloco**. Sexo e UF já vêm em cada voto;
telefones, e-mails, URLs, suplentes e cargos não entram na análise.

Os **blocos parlamentares** reúnem partidos que atuam juntos no Senado. Cada partido pertence a um
único bloco, então derivamos o mapa `partido → bloco` dos senadores atuais. Isso permite atribuir
bloco também a quem votou em 2025 mas **já não está em exercício**, a partir do partido registrado
no voto.

In [8]:
P = "identificacao_parlamentar_"
COLUNAS_SENADOR: dict[str, str] = {
    f"{P}codigo_parlamentar": "codigo_senador",
    f"{P}nome_parlamentar": "nome",
    f"{P}sigla_partido_parlamentar": "partido",
    f"{P}bloco_nome_bloco": "bloco",
}

senadores, relatorio = limpar(pd.json_normalize(senadores_raw), colunas=COLUNAS_SENADOR,
                              chave_duplicata=[f"{P}codigo_parlamentar"])
mostrar(relatorio)

# 'Bloco Parlamentar da Resistência Democrática' -> 'Resistência Democrática'
senadores["bloco"] = (senadores["bloco"].str.removeprefix("Bloco Parlamentar ")
                      .str.removeprefix("da ").str.removeprefix("do "))

com_bloco = senadores.dropna(subset=["bloco"])
if com_bloco.groupby("partido")["bloco"].nunique().gt(1).any():  # premissa: um bloco por partido
    log.warning("Há partido em mais de um bloco; o mapa partido -> bloco usará o primeiro")
PARTIDO_BLOCO: dict[str, str] = com_bloco.drop_duplicates("partido").set_index("partido")["bloco"].to_dict()

print(com_bloco.groupby("bloco")["partido"].agg(lambda s: ", ".join(sorted(set(s)))).to_string(), end="\n\n")
senadores.head()

  · entrada: 81 linhas x 33 colunas
  · removidas 29 colunas fora da seleção
  · 'identificacao_parlamentar_codigo_parlamentar' -> inteiro
  · saída: 81 linhas x 4 colunas

bloco
Aliança                          PP, REPUBLICANOS
Democracia                 MDB, PODE, PSDB, UNIÃO
Pelo Brasil                               PDT, PT
Resistência Democrática                  PSB, PSD
Vanguarda                              AVANTE, PL



,codigo_senador,nome,partido,bloco
0,5672,Alan Rick,REPUBLICANOS,Aliança
1,5982,Alessandro Vieira,MDB,Democracia
2,6358,Ana Paula Lobato,PSB,Resistência Democrática
3,5967,Angelo Coronel,REPUBLICANOS,Aliança
4,6009,Astronauta Marcos Pontes,PL,Vanguarda


### 4.3 Votações

Das 38 colunas, mantemos 10. Ficam de fora o texto longo de tramitação (`informe_legislativo`),
códigos internos, o ano da matéria (redundante com a identificação, ex.: `PEC 81/2015`) e duas
colunas constantes: todas as votações são do Senado (`SF`), em sessão deliberativa ordinária (`DOR`).

Os totais de votos só vêm preenchidos nas votações **secretas**. Nas abertas, eles são calculados a
partir dos votos individuais (seção 4.4).

In [9]:
TIPOS_MATERIA: dict[str, str] = {
    "MSF": "Mensagem (indicação de autoridade)",
    "OFS": "Ofício (indicação de autoridade)",
    "PEC": "Proposta de Emenda à Constituição",
    "PLP": "Projeto de Lei Complementar",
    "PL": "Projeto de Lei",
    "MPV": "Medida Provisória",
    "RQS": "Requerimento",
}
COLUNAS_VOTACAO: dict[str, str] = {
    "codigo_sessao_votacao": "id_votacao",
    "data_sessao": "data",
    "sigla": "tipo_materia",
    "identificacao": "materia",
    "descricao_votacao": "descricao",
    "ementa": "ementa",
    "id_processo": "id_processo",
    "votacao_secreta": "secreta",
    "resultado_votacao": "resultado",
    "total_votos_sim": "total_sim",
    "total_votos_nao": "total_nao",
    "total_votos_abstencao": "total_abstencao",
}
TOTAIS: list[str] = ["total_sim", "total_nao", "total_abstencao"]

votacoes, relatorio = limpar(pd.json_normalize(votacoes_raw), colunas=COLUNAS_VOTACAO,
                             chave_duplicata=["codigo_sessao_votacao"])
mostrar(relatorio)

# regras do domínio
votacoes["aprovada"] = votacoes.pop("resultado").eq("A")
votacoes["tipo_materia_desc"] = votacoes["tipo_materia"].map(TIPOS_MATERIA).fillna(votacoes["tipo_materia"])
votacoes = votacoes.sort_values("data").reset_index(drop=True)
votacoes.head()

  · entrada: 128 linhas x 38 colunas
  · removidas 26 colunas fora da seleção
  · 'total_votos_abstencao' -> inteiro (com nulos)
  · 'total_votos_nao' -> inteiro (com nulos)
  · 'total_votos_sim' -> inteiro (com nulos)
  · 'data_sessao' -> data
  · 'votacao_secreta' -> booleano
  · saída: 128 linhas x 12 colunas



,id_votacao,data,descricao,ementa,id_processo,materia,tipo_materia,total_abstencao,total_nao,total_sim,secreta,aprovada,tipo_materia_desc
0,6918,2025-02-19,Votação nominal da Emenda nº 1 (Substitutivo) ...,Dispõe sobre prazo para liquidação de restos a...,8793330,PLP 22/2025,PLP,<NA>,<NA>,<NA>,False,True,Projeto de Lei Complementar
1,6919,2025-03-11,Votação nominal da Proposta de Emenda à Consti...,Dá nova redação ao § 4º do art. 225 da Constit...,8664217,PEC 18/2024,PEC,<NA>,<NA>,<NA>,False,True,Proposta de Emenda à Constituição
2,6920,2025-03-11,Votação nominal da Proposta de Emenda à Consti...,Dá nova redação ao § 4º do art. 225 da Constit...,8664217,PEC 18/2024,PEC,<NA>,<NA>,<NA>,False,True,Proposta de Emenda à Constituição
3,6921,2025-03-18,Votação nominal da decisão da Presidência na Q...,"Altera a Lei Complementar nº 64, de 18 de maio...",8544415,PLP 192/2023,PLP,<NA>,<NA>,<NA>,False,False,Projeto de Lei Complementar
4,6922,2025-03-19,Votação nominal do Substitutivo da Câmara dos ...,Dispõe sobre extensão do prazo para liquidação...,8803985,PLP 22/2025 (Substitutivo-CD),PLP,<NA>,<NA>,<NA>,False,True,Projeto de Lei Complementar


### 4.4 Votos individuais

Os votos vêm dentro de cada votação; `json_normalize` os transforma numa linha por senador por votação.

A API usa 12 códigos que misturam **voto** (Sim, Não), **presença sem voto** e **motivos de ausência**.
Eles são agrupados em categorias e numa coluna booleana `presente`. Nas votações secretas o código é
sempre `Votou`: sabemos **quem** votou, mas não **como**.

In [10]:
CATEGORIAS_VOTO: dict[str, str] = {
    "Sim": "Sim",
    "Não": "Não",
    "Abstenção": "Abstenção",
    "Votou": "Votou (secreta)",
    "Presidente (art. 51 RISF)": "Presidindo a sessão",
    "P-NRV": "Presente sem voto",
    "AP": "Ausência justificada",   # atividade parlamentar
    "MIS": "Ausência justificada",  # missão oficial no país ou exterior
    "LS": "Licença",                # licença saúde
    "LP": "Licença",                # licença particular
    "NCom": "Ausente sem justificativa",
    "NA": "Não se aplica",          # dispositivo não citado na votação
}
PRESENTES: set[str] = {"Sim", "Não", "Abstenção", "Votou (secreta)", "Presidindo a sessão", "Presente sem voto"}
COLUNAS_VOTO: dict[str, str] = {
    "codigo_sessao_votacao": "id_votacao",
    "codigo_parlamentar": "codigo_senador",
    "nome_parlamentar": "nome",
    "sigla_partido_parlamentar": "partido",
    "sigla_uf_parlamentar": "uf",
    "sexo_parlamentar": "sexo",
    "sigla_voto_parlamentar": "voto",
}

votos, relatorio = limpar(
    pd.json_normalize(votacoes_raw, record_path="votos", meta=["codigoSessaoVotacao"]),
    colunas=COLUNAS_VOTO,                     # descarta a descrição por extenso do código, redundante
    manter_texto=["sigla_voto_parlamentar"],  # protege o código 'NA'
    chave_duplicata=["codigo_sessao_votacao", "codigo_parlamentar"],
)
mostrar(relatorio)
votos = votos[["id_votacao", *votos.columns.drop("id_votacao")]]  # chave primeiro

# regras do domínio
votos["categoria"] = votos["voto"].map(CATEGORIAS_VOTO)
desconhecidos = votos.loc[votos["categoria"].isna(), "voto"].unique()
if len(desconhecidos):
    log.warning("Códigos de voto não mapeados: %s", list(desconhecidos))
    votos["categoria"] = votos["categoria"].fillna("Outro")
votos["presente"] = votos["categoria"].isin(PRESENTES)
# partido na data do voto -> bloco (NOVO e sem partido não integram bloco)
votos["bloco"] = votos["partido"].map(PARTIDO_BLOCO).fillna("Sem bloco")

# totais das votações abertas, calculados a partir dos votos
contagem = pd.crosstab(votos["id_votacao"], votos["voto"])
for coluna, voto in zip(TOTAIS, ["Sim", "Não", "Abstenção"]):
    votacoes[coluna] = votacoes[coluna].fillna(votacoes["id_votacao"].map(contagem[voto]))

print(votos["categoria"].value_counts().to_string(), end="\n\n")
votos.head()

  · entrada: 10368 linhas x 8 colunas
  · removidas 1 colunas fora da seleção
  · 'codigo_sessao_votacao' -> inteiro
  · saída: 10368 linhas x 7 colunas

categoria
Votou (secreta)              3774
Sim                          3023
Presente sem voto            1544
Ausência justificada         1012
Não                           549
Licença                       340
Presidindo a sessão            59
Ausente sem justificativa      37
Abstenção                      15
Não se aplica                  15



,id_votacao,codigo_senador,nome,sexo,partido,uf,voto,categoria,presente,bloco
0,6918,5672,Alan Rick,M,UNIÃO,AC,Sim,Sim,True,Democracia
1,6918,5982,Alessandro Vieira,M,MDB,SE,Sim,Sim,True,Democracia
2,6918,6358,Ana Paula Lobato,F,PDT,MA,AP,Ausência justificada,False,Pelo Brasil
3,6918,5967,Angelo Coronel,M,PSD,BA,AP,Ausência justificada,False,Resistência Democrática
4,6918,6009,Astronauta Marcos Pontes,M,PL,SP,AP,Ausência justificada,False,Vanguarda


### 4.5 Autoria das propostas

O autor de cada proposta é classificado em um **grupo de autoria**:

| Tipo de autor na API | Grupo |
|---|---|
| Presidente da República | Governo federal |
| Deputado, Câmara dos Deputados | Câmara |
| Senador, líder partidário, presidente do Senado | Senado |
| STF, Procuradoria-Geral, outros entes | Outros órgãos |

Para propostas do Senado, identificamos também o **bloco do autor**. O partido usado é o de 2025
(registrado nos votos), pois a autoria guarda o partido da época da apresentação: a PEC 81/2015, por
exemplo, aparece com o DEM, partido extinto em 2022. Propostas coletivas cujos autores são de blocos
diferentes (comum em PECs, que exigem 27 assinaturas) ficam como **"Vários blocos"**.

In [11]:
GRUPO_AUTORIA: dict[str, str] = {
    "PRESIDENTE_REPUBLICA": "Governo federal",
    "DEPUTADO": "Câmara",
    "CAMARA": "Câmara",
    "SENADOR": "Senado",
    "LIDER": "Senado",
    "PRESIDENTE_SF": "Senado",
}
COLUNAS_AUTOR: dict[str, str] = {
    "id_processo": "id_processo",
    "autor": "autor",
    "sigla_tipo": "tipo_autor",
    "codigo_parlamentar": "codigo_parlamentar",
    "sigla_partido": "partido",
}

# uma linha por autor, na ordem da API (o primeiro é o autor principal)
autores, relatorio = limpar(
    pd.json_normalize([{**a, "id_processo": id_processo}
                       for id_processo, lista in autorias_raw.items() for a in lista]),
    colunas=COLUNAS_AUTOR,
)
mostrar(relatorio)

partido_2025 = votos.groupby("codigo_senador")["partido"].agg(lambda s: s.mode()[0])
autores["grupo"] = autores["tipo_autor"].map(GRUPO_AUTORIA).fillna("Outros órgãos")
autores["partido"] = autores["codigo_parlamentar"].map(partido_2025).fillna(autores["partido"])
autores["bloco"] = autores["partido"].map(PARTIDO_BLOCO).fillna("Sem bloco").where(autores["grupo"].eq("Senado"))


def bloco_da_autoria(blocos: pd.Series) -> str | None:
    """Bloco comum aos autores do Senado; 'Vários blocos' se houver mais de um."""
    distintos = set(blocos.dropna())
    return distintos.pop() if len(distintos) == 1 else ("Vários blocos" if distintos else None)


principais = autores.drop_duplicates("id_processo").set_index("id_processo")
autorias = pd.DataFrame({
    "autor_principal": principais["autor"],
    "tipo_autor": principais["tipo_autor"],
    "partido_autor": principais["partido"],
    "grupo_autoria": principais["grupo"],
    "n_autores": autores.groupby("id_processo").size(),
    "bloco_autoria": autores.groupby("id_processo")["bloco"].agg(bloco_da_autoria),
}).reset_index()
# o bloco só se aplica quando o autor principal é do Senado
autorias["bloco_autoria"] = autorias["bloco_autoria"].where(autorias["grupo_autoria"].eq("Senado"))

votacoes = votacoes.merge(autorias, on="id_processo", how="left")
votacoes["grupo_autoria"] = votacoes["grupo_autoria"].fillna("Sem autoria informada")

print(votacoes.groupby(["grupo_autoria", "secreta"]).size().unstack(fill_value=0)
      .rename(columns={False: "abertas", True: "secretas"}).to_string(), end="\n\n")
print(votacoes.loc[~votacoes["secreta"] & votacoes["grupo_autoria"].eq("Senado"), "bloco_autoria"]
      .value_counts().to_string())

  · entrada: 358 linhas x 17 colunas
  · removidas 12 colunas fora da seleção
  · 'codigo_parlamentar' -> inteiro (com nulos)
  · saída: 358 linhas x 5 colunas

secreta          abertas  secretas
grupo_autoria                     
Câmara                16         2
Governo federal        6        56
Outros órgãos          1        13
Senado                33         1

bloco_autoria
Vários blocos              22
Pelo Brasil                 3
Democracia                  3
Resistência Democrática     3
Aliança                     1
Vanguarda                   1


### 4.6 Checagens de qualidade

Verificações automáticas antes da análise: chaves únicas, ausência de nulos nas colunas essenciais
e consistência entre as tabelas.

In [12]:
secretas_com_voto_aberto = (votos.merge(votacoes[["id_votacao", "secreta"]], on="id_votacao")
                            .query("secreta and categoria in ['Sim', 'Não', 'Abstenção']"))
checagens: dict[str, bool] = {
    "id_votacao único em votacoes": votacoes["id_votacao"].is_unique,
    "um voto por senador por votação": not votos.duplicated(["id_votacao", "codigo_senador"]).any(),
    "todo voto pertence a uma votação": votos["id_votacao"].isin(votacoes["id_votacao"]).all(),
    "sem nulos em colunas essenciais": not votos[["id_votacao", "codigo_senador", "partido", "categoria"]].isna().any().any(),
    "código de voto 'NA' preservado": votos["voto"].eq("NA").any(),
    "votações secretas sem voto aberto": secretas_com_voto_aberto.empty,
    "todas as votações têm autoria": votacoes["grupo_autoria"].ne("Sem autoria informada").all(),
    "todas as votações têm totais": votacoes[TOTAIS[:2]].notna().all().all(),
}
for nome, ok in checagens.items():
    print(f"{'OK   ' if ok else 'FALHA'} {nome}")

fora = set(votos["codigo_senador"]) - set(senadores["codigo_senador"])
print(f"\nSenadores distintos nos votos: {votos['codigo_senador'].nunique()} ({len(fora)} fora de exercício hoje)")
trocas = votos.groupby(["codigo_senador", "nome"])["partido"].unique().loc[lambda s: s.str.len() > 1]
print(f"Trocaram de partido durante 2025: {len(trocas)}")
print(trocas.apply(" → ".join).droplevel(0).to_string())

OK    id_votacao único em votacoes
OK    um voto por senador por votação
OK    todo voto pertence a uma votação
OK    sem nulos em colunas essenciais
OK    código de voto 'NA' preservado
OK    votações secretas sem voto aberto
OK    todas as votações têm autoria
OK    todas as votações têm totais

Senadores distintos nos votos: 84 (10 fora de exercício hoje)
Trocaram de partido durante 2025: 5
nome
Marcio Bittar                  UNIÃO → PL
Alan Rick            UNIÃO → REPUBLICANOS
Daniella Ribeiro     PSD → S/Partido → PP
Giordano                  MDB → S/Partido
Margareth Buzetti    PSD → S/Partido → PP


### 4.7 Tabelas finais

Por último, colunas de texto com poucos valores distintos (partido, UF, bloco, categoria de voto)
viram `category`. Isso fica para o fim porque operações como `fillna` com valores novos falham em
colunas categóricas.

In [13]:
tabelas: dict[str, pd.DataFrame] = {"senadores": senadores, "votacoes": votacoes, "votos": votos}
for nome, tabela in tabelas.items():
    categorizar(tabela)
    print(f"{nome}: {tabela.shape[0]} linhas x {tabela.shape[1]} colunas")

resumo(votos)

senadores: 81 linhas x 4 colunas
votacoes: 128 linhas x 19 colunas
votos: 10368 linhas x 10 colunas


,tipo,nulos_%,distintos,exemplo
id_votacao,Int64,0.0,128,6918
codigo_senador,int64,0.0,84,5672
nome,string,0.0,84,Alan Rick
sexo,category,0.0,2,M
partido,category,0.0,13,UNIÃO
uf,category,0.0,27,AC
voto,category,0.0,12,Sim
categoria,category,0.0,10,Sim
presente,bool,0.0,2,True
bloco,category,0.0,6,Democracia


In [14]:
resumo(votacoes)

,tipo,nulos_%,distintos,exemplo
id_votacao,int64,0.0,128,6918
data,datetime64[us],0.0,36,2025-02-19 00:00:00
descricao,string,0.0,127,Votação nominal da Emenda nº 1 (Substitutivo) ...
ementa,string,0.0,111,Dispõe sobre prazo para liquidação de restos a...
id_processo,int64,0.0,111,8793330
materia,string,0.0,111,PLP 22/2025
tipo_materia,category,0.0,7,PLP
total_abstencao,Int64,0.0,4,0
total_nao,Int64,0.0,27,1
total_sim,Int64,0.0,39,65
